# SpaceX Falcon 9 - Web Scraping
IBM Applied Data Science Capstone

In [ ]:
import sys
import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd

In [ ]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"
response = requests.get(static_url)
print("Status code:", response.status_code)

In [ ]:
soup = BeautifulSoup(response.text, 'html.parser')
print(soup.title)

In [ ]:
html_tables = soup.find_all('table')
print("Number of tables found:", len(html_tables))

In [ ]:
# The target table is usually the third one (index 2)
first_launch_table = html_tables[2]
print(first_launch_table.prettify()[:500])

In [ ]:
column_names = []
for th in first_launch_table.find_all('th'):
    name = th.get_text(strip=True)
    if name:
        column_names.append(name)
print(column_names)

In [ ]:
launch_dict = dict.fromkeys(column_names)
del launch_dict['Date and time ( )']

launch_dict['Flight No.'] = []
launch_dict['Launch site'] = []
launch_dict['Payload'] = []
launch_dict['Payload mass'] = []
launch_dict['Orbit'] = []
launch_dict['Customer'] = []
launch_dict['Launch outcome'] = []
launch_dict['Version Booster'] = []
launch_dict['Booster landing'] = []
launch_dict['Date'] = []
launch_dict['Time'] = []

In [ ]:
extracted_row = 0
for table_number, table in enumerate(soup.find_all('table', "wikitable plainrowheaders collapsible")):
    for row in table.find_all("tr"):
        if row.th:
            if row.th.string:
                flight_number = row.th.string.strip()
                flag = flight_number.isdigit()
        else:
            flag = False
        row_data = row.find_all('td')
        if flag:
            extracted_row += 1
            # Flight Number
            launch_dict['Flight No.'].append(flight_number)
            # Date
            date_time = row_data[0].get_text(strip=True)
            date = date_time[:-7] if len(date_time) > 7 else date_time
            launch_dict['Date'].append(date)
            # Time
            time = date_time[-7:] if len(date_time) > 7 else ''
            launch_dict['Time'].append(time)
            # Booster version
            bv = row_data[1].a.string if row_data[1].a else row_data[1].get_text(strip=True)
            launch_dict['Version Booster'].append(bv)
            # Launch Site
            launch_dict['Launch site'].append(row_data[2].get_text(strip=True))
            # Payload
            launch_dict['Payload'].append(row_data[3].get_text(strip=True))
            # Payload Mass
            launch_dict['Payload mass'].append(row_data[4].get_text(strip=True))
            # Orbit
            launch_dict['Orbit'].append(row_data[5].get_text(strip=True))
            # Customer
            launch_dict['Customer'].append(row_data[6].get_text(strip=True))
            # Launch outcome
            launch_dict['Launch outcome'].append(row_data[7].get_text(strip=True))
            # Booster landing
            launch_dict['Booster landing'].append(row_data[8].get_text(strip=True))

print("Rows extracted:", extracted_row)

In [ ]:
df = pd.DataFrame({key: pd.Series(value) for key, value in launch_dict.items()})
df.head()

In [ ]:
df.to_csv('spacex_web_scraped.csv', index=False)
print("spacex_web_scraped.csv saved successfully")